# 4. Connectors and Serialization

- Now that we know how to process data inside the Flink "bubble," we need to learn how to get data in (Sources) and out (Sinks). This section covers the mechanics of external system integration and how Flink ensures data is readable across the network

## Anatomy of a Connector

- In Flink, a connector is more than just a driver; it is a bridge that must respect Flink's checkpointing and partitioning logic.
    - Sources: Responsible for reading data from external systems.
    - Sinks: Responsible for writing data to external systems.

- With more recent versions of Flink, Dynamic Discovery is now possible. Modern flink connectors using the Source and SinkV2 APIs can dynamically discover new partitions (e.g., new Kafka partitions) without a job restart!

### Kafka

- Kafka is the most common companion for Flink. It is often used as the "Source of Truth" for streaming applications

- Kafka Source: Kafka is designed to be highly parallel, and each Flink sub-task reads from a preassigned set of Kafka partitions. 
    - This is quite important to note! Flink is not super reactive; once these partitions are set, if there are uneven partitions, you need to restart the job to redistribute the partition data flows! 
    - Even if Flink claims to reactively manage this for you, it STILL requires a restart of the app before the repartitioning occurs
    - This is something to note, because if you have some critical consumer downstream, reactive partitioning means unexpected downtime

    ```java
    KafkaSource<String> source = KafkaSource.<String>builder()
        .setBootstrapServers("localhost:9092")
        .setTopics("input-topic")
        .setGroupId("my-flink-group")
        .setStartingOffsets(OffsetsInitializer.earliest())
        .setValueOnlyDeserializer(new SimpleStringSchema())
        .build();

    DataStream<String> stream = env.fromSource(source, WatermarkStrategy.noWatermarks(), "Kafka Source");
    ```

- Kafka Sink: To ensure that Flink doesn't write duplicate data to Kafka during a crash/recovery cycle, it uses a Two-Phase Commit protocol
    - Pre-commit: Flink writes data to Kafka but marks it as "uncommitted."
    - Commit: Once a Flink checkpoint is successful, Flink tells Kafka to mark those records as "committed."

## Serialisation

- Because Flink is a distributed system, data must be turned into bytes to move between TaskManagers. This is called **Serialization**.

- Flink is fastest when your data objects are "POJOs" (Plain Old Java Objects). For Flink to recognize a class as a POJO and optimize it, it must meet these criteria:
    - The class must be public and standalone.
    - It must have a public no-argument constructor.
    - All fields must be either public or have standard getters and setters.
    - Field types must be supported by Flink's serialization.

## Looking up external databases: Always use Async!!!

- One very common use case in Flink is to enrich your data somehow using an external database like JDBC or Redis

- Connecting to databases is expensive business; if you don't manage your database connection properly (e.g. with `Rich*` operators), you will crash your DB, or your Flink app, or both

- Moreover, you want to ensure that when you connect to the database, your database calls do not block the thread in the TaskSlot
    - If you write a database call using `Map`, your Flink app will be extremely slow
    - Why? Because the thread making the call is blocked until the database returns!!!
    - This is why Async is the preferred way; always allow Flink to send multiple requests to a database simultaneously without waiting for the first one to finish
    - To do this, use `AsyncDataStream` and implement an `AsyncFunction`

## File Sinks (S3/HDFS)

- Writing to files in a streaming world is tricky because you can't have thousands of tiny files. Flink uses the FileSink to manage this
    - Partially Written Files: Flink writes to .inprogress files
    - Rolling Policy: You define when a file is "finished" (e.g., every 10 minutes or every 100MB)
    - Bucketing: You can organize files by time or key (e.g., s3://my-bucket/year=2026/month=03/day=22/)